In [7]:
from data_handling.batch_loader_np import DynamicBatchLoaderNp
from data_handling.tokenizer import CharacterTokenizer
from layers.oracle import PicoGPTOracle
from layers.runner import Runner

VOCAB_STR = (" \n\t0123456789abcdefghijklmnopqrstuvwxyz"
             "ABCDEFGHIJKLMNOPQRSTUVWXYZ.,?!;:'\"-—…()[]{}*_&$%/\\Code“”‘’")

def build(tokenizer, T, C, L, NH):
    return PicoGPTOracle(vocab_size=tokenizer.vocab_size, d_model=C,
                         n_layers=L, n_heads=NH, max_seq_len=T)


def train():
    tokenizer = CharacterTokenizer(VOCAB_STR)
    B, T = 32, 128
    C, L, NH = 256, 4, 8
    loader = DynamicBatchLoaderNp("../notebooks/data", B, T,
                                swap_every_iterations=100,
                                char_to_int=tokenizer.stoi)
    model = build(tokenizer, T, C, L, NH)
    runner = Runner(model, loader, tokenizer, B, T,
                    max_steps=2000, eval_interval=200)
    runner.train()

In [9]:
from data_handling.chunker_loader import ChunkerLoader

chunker_loader = ChunkerLoader(stories_per_chunk=10000, max_chunks=10)
chunker_loader.create_chunks()

/home/maksim/PycharmProjects/pico-gpt/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TinyStories dataset...


Streaming and splitting data into chunks of 10000 stories...
Saved: data/train_chunk_1.txt (52631 raw text lines)
Saved: data/train_chunk_2.txt (53931 raw text lines)
Saved: data/train_chunk_3.txt (50318 raw text lines)
Saved: data/train_chunk_4.txt (54721 raw text lines)
Saved: data/train_chunk_5.txt (50846 raw text lines)
Saved: data/train_chunk_6.txt (53296 raw text lines)
Saved: data/train_chunk_7.txt (55752 raw text lines)
Saved: data/train_chunk_8.txt (50745 raw text lines)
Saved: data/train_chunk_9.txt (48915 raw text lines)
Saved: data/train_chunk_10.txt (51351 raw text lines)
Saving validation set to data/tiny_stories_val.txt...


In [10]:
train()

Starting standalone execution loop pipeline...
[Data Loader] train iter 0: loading ../notebooks/data/train_chunk_1.txt
[Data Loader] val iter 0: loading ../notebooks/data/tiny_stories_val.txt
Step    0 | Train Loss: 4.8443 | Val Loss: 4.8564
--- Sampling Snapshot: ---
O%UOl*,u}%q},a*6,a]buZaw''}H([(,uaz‘…‘xa[
--------------------------
[Data Loader] train iter 100: loading ../notebooks/data/train_chunk_10.txt
[Data Loader] train iter 200: loading ../notebooks/data/train_chunk_2.txt
Step  200 | Train Loss: 2.1175 | Val Loss: 2.1398
--- Sampling Snapshot: ---
On The a heda frinenene pimen ad hevely w
--------------------------
[Data Loader] train iter 300: loading ../notebooks/data/train_chunk_3.txt
[Data Loader] train iter 400: loading ../notebooks/data/train_chunk_4.txt
Step  400 | Train Loss: 1.7431 | Val Loss: 1.8712
--- Sampling Snapshot: ---
One the a big ther for cames. 
One day, a
--------------------------
[Data Loader] train iter 500: loading ../notebooks/data/train_chunk_5.txt

In [22]:
import numpy as np
def infer(checkpoint="pico_gpt_oracle_np", temperature=0.65, max_new_tokens=250):
    tokenizer = CharacterTokenizer(VOCAB_STR)
    B, T = 32, 128
    C, L, NH = 256, 4, 8
    model = build(tokenizer, T, C, L, NH)
    print(f"Loading weights from '{checkpoint}.npz'...")
    model.load(checkpoint)
    # model.fuse_norms_for_inference()
    runner = Runner(model, batch_loader=None, tokenizer=tokenizer, B=B, T=T)

    prompt = ""
    while prompt != "q":
        prompt = input("Prompt: ")
        if prompt == "q":
            break
        ctx = np.array([tokenizer.encode(prompt)], dtype=np.int64)
        out = runner.generate_text(ctx, max_new_tokens=max_new_tokens,
                                   block_size=T, temperature=temperature)
        print("\n--- Model Output ---")
        print(tokenizer.decode(out[0].tolist()))
        print("--------------------")

In [23]:
infer()

Loading weights from 'pico_gpt_oracle_np.npz'...

--- Model Output ---
Anna said, "I live you some in the toys."
The man says, "I like this pus. You have you littttle bry saying our plant well. They have to lid and to the bird. You are helped you home. They have drom and say is he was lot?"
The bear was time and smiled. They smi
--------------------


KeyboardInterrupt: Interrupted by user